In [28]:
import os
import json
import pandas as pd
import torch
import clip
from PIL import Image
from tqdm import tqdm
from collections import defaultdict

device = "cuda" if torch.cuda.is_available() else "cpu"

train_dir = "dataset/train"
test_dir = "dataset/test"

model, preprocess = clip.load("ViT-B/32", device=device)

In [29]:
word_freq = defaultdict(int)

for folder in os.listdir(train_dir):
    metadata_path = os.path.join(train_dir, folder, "metadata.json")
    with open(metadata_path, "r") as f:
        metadata = json.load(f)
    for word in metadata['correct_words']:
        word_freq[word] += 1

In [34]:
def get_image_embedding(image_path):
    image = preprocess(Image.open(image_path).convert("RGB")).unsqueeze(0).to(device)
    with torch.no_grad():
        embedding = model.encode_image(image)
    embedding /= embedding.norm(dim=-1, keepdim=True)
    return embedding

def get_text_embeddings(text_list):
    text_tokens = clip.tokenize(text_list).to(device)
    with torch.no_grad():
        embeddings = model.encode_text(text_tokens)
    embeddings /= embeddings.norm(dim=-1, keepdim=True)
    return embeddings

def select_top5_words(image_path, word_choices):
    image_emb = get_image_embedding(image_path)
    text_embs = get_text_embeddings(word_choices)
    
    similarity = (image_emb @ text_embs.T).squeeze(0).cpu().numpy()
    
    rerank_score = similarity.copy()
    for i, w in enumerate(word_choices):
        rerank_score[i] += 0.05 * word_freq.get(w, 0)
    
    top5_indices = rerank_score.argsort()[::-1][:5]
    top5_words = [word_choices[i] for i in top5_indices]
    
    return top5_words
    
    return top5_words

In [35]:
submission = []

for folder in tqdm(sorted(os.listdir(test_dir))):
    metadata_path = os.path.join(test_dir, folder, "metadata.json")
    
    with open(metadata_path, "r") as f:
        metadata = json.load(f)
    
    image_path = metadata["image_path"]
    word_choices = metadata["word_choices"]
    
    top5_words = select_top5_words(image_path, word_choices)
    
    submission.append({
        "ID": folder,
        "word1": top5_words[0],
        "word2": top5_words[1],
        "word3": top5_words[2],
        "word4": top5_words[3],
        "word5": top5_words[4]
    })

100%|████████████████████████████████████████████████████████████████████████████████| 100/100 [00:34<00:00,  2.92it/s]


In [36]:
submission_df = pd.DataFrame(submission)
submission_df.to_csv("submission.csv", index=False)

In [37]:
submission_df.head()

,ID,word1,word2,word3,word4,word5
0,00000,paper,marker,desk_lamp,aura,duck
1,00001,book,pencil,grimoire,warding,broccoli
2,00002,book,eraser,newt,imp,kangaroo
3,00003,projector,pen,desk_lamp,lamp,water
4,00004,comb,tablet,vacuum,faucet,seaotter
